# 🚀 MeshGraphNet 3D FEM Surrogate Model Training
Train the **MeshGraphNet** surrogate model on 3D FEM structural simulation data (quadratic 10-node tetrahedra).

### Features:
- **Tet10 nodal strain physics bridge:** Evaluates strain at each node's natural coordinate (restores corner sensitivity).
- **Adaptive Loss Balancer:** Live EMA rebalancing across displacement, stress, strain, and von Mises loss terms.
- **Inverse Hooke's law strain derivation:** Automatically populates ground-truth strain from stress.
- **Base-sample-aware train/val/test splitting:** Zero variant leakage across splits.

## 1. Environment Setup

In [ ]:
# Install PyG and scientific dependencies
!pip install torch torch_geometric h5py meshio pyyaml pandas matplotlib -q
print("✓ Dependencies installed successfully!")

## 2. Mount Google Drive & Configure Paths

In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')

# Adjust these paths to your Google Drive directory:
PROJECT_ROOT = '/content/drive/MyDrive/FEM_GNN_Project'
H5_DIR       = f'{PROJECT_ROOT}/output/raw'
MANIFEST     = f'{PROJECT_ROOT}/output/manifest.csv'
GNN_CODE     = f'{PROJECT_ROOT}/gnn_project_version_2'
BRIDGE_CODE  = f'{PROJECT_ROOT}/Training_Pipeline_version_2'

sys.path.insert(0, GNN_CODE)
sys.path.insert(0, BRIDGE_CODE)
print("✓ Added to sys.path!")

## 3. Check Hardware & Device

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Active device:", device)

## 4. Auto-Split Manifest & Create DataLoaders

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Material properties (Domex 420MC structural steel)
MAT_E = 210.0e9
MAT_NU = 0.3
MAT_YIELD = 420.0e6
BATCH_SIZE = 4

manifest_file = Path(MANIFEST)
if manifest_file.exists():
    manifest_df = pd.read_csv(manifest_file)
    if "split" not in manifest_df.columns:
        print("⚠ 'split' column missing. Auto-assigning base-sample splits...")
        base_ids = manifest_df["base_sample_id"].unique()
        rng = np.random.default_rng(42)
        rng.shuffle(base_ids)
        n_train = max(1, int(0.8 * len(base_ids)))
        n_val = max(1, int(0.1 * len(base_ids)))
        train_bases = set(base_ids[:n_train])
        val_bases = set(base_ids[n_train:n_train + n_val])
        manifest_df["split"] = manifest_df["base_sample_id"].apply(
            lambda b: "train" if b in train_bases else ("val" if b in val_bases else "test")
        )
        manifest_df.to_csv(manifest_file, index=False)
        print("✓ Saved split column:", manifest_df["split"].value_counts().to_dict())

from gnn_bridge.dataloader import create_dataloaders, FEMGraphDataset, compute_field_stds

loaders = create_dataloaders(h5_dir=H5_DIR, manifest_path=MANIFEST, batch_size=BATCH_SIZE, E=MAT_E, nu=MAT_NU)
train_dataset = FEMGraphDataset(h5_dir=H5_DIR, manifest_path=MANIFEST, split="train", E=MAT_E, nu=MAT_NU)
field_stds = compute_field_stds(train_dataset)
print("✓ Field normalisation stds:", field_stds)

## 5. Build Model & Multi-Term Loss

In [ ]:
from models.meshgraphnet import MeshGraphNet
from models.loss import MeshGraphNetLoss

model = MeshGraphNet(
    node_in_dim=11,
    edge_in_dim=4,
    global_in_dim=1,
    hidden_dim=128,
    num_processor_layers=15,
    stress_net_local_mp_layers=3,
    E=MAT_E,
    nu=MAT_NU,
    yield_strength=MAT_YIELD,
)

loss_fn = MeshGraphNetLoss(
    field_stds=field_stds,
    w_u=1.0,
    w_sigma=0.5,
    w_eps=0.3,
    w_eps_corr=0.3,
    w_vm=0.2,
)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ MeshGraphNet initialised with {n_params:,} trainable parameters!")

## 6. Run Training with Live Adaptive Loss Balancing

In [ ]:
from gnn_bridge.trainer import Trainer

NUM_EPOCHS = 100
LR = 1e-3

trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    train_loader=loaders["train"],
    val_loader=loaders.get("val"),
    lr=LR,
    grad_clip_norm=1.0,
    device=str(device),
    checkpoint_dir=f"{PROJECT_ROOT}/checkpoints",
    log_dir=f"{PROJECT_ROOT}/logs",
    adaptive_loss_weighting=True,
)

print("Starting training...")
history = trainer.train(num_epochs=NUM_EPOCHS)

## 7. Plot Loss Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history["train_loss"], label="Train Loss", linewidth=2)
if history["val_loss"]:
    axes[0].plot(history["val_loss"], label="Val Loss", linewidth=2)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Total Loss")
axes[0].set_title("Training and Validation Loss")
axes[0].legend()
axes[0].set_yscale("log")
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["lr"], linewidth=2, color="green")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("Learning Rate Schedule")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Final Evaluation on Validation and Test Sets

In [ ]:
from gnn_bridge.metrics import evaluate_model, print_evaluation_report

best_ckpt = f"{PROJECT_ROOT}/checkpoints/best_model.pt"
if os.path.exists(best_ckpt):
    trainer.load_checkpoint(best_ckpt)
    print("Loaded best checkpoint!")

for split_name in ("val", "test"):
    if split_name in loaders:
        results = evaluate_model(model, loaders[split_name], device=str(device))
        print_evaluation_report(results, split_name)